<a href="https://colab.research.google.com/github/tadapin/tidb-colab-tutorials/blob/main/tutorials/05_hybrid_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ハイブリッド検索 (EC 商品カタログ)

このノートブックは **pytidb シリーズの第 5 回** です。ベクトル検索と全文検索の結果を融合 (fusion) する「ハイブリッド検索」を扱います。

## 学習目標
- 1 つのテーブルに `VectorField` と `FullTextField` を共存させる
- `search_type="hybrid"` でハイブリッド検索を行う
- `fusion` のアルゴリズム (`rrf` / `weighted`) を切り替える
- ベクトル単独・全文単独・ハイブリッドの結果を見比べる

前提: `03_vector_search.ipynb` と `04_fulltext_search.ipynb` を実行済み。


## 1. 依存と接続


In [ ]:
!pip install -q pytidb


In [ ]:
from pathlib import Path
import requests

# TiDB Cloud Zero のAPIエンドポイント (サインアップ不要、30日有効)
ZERO_API = "https://zero.tidbapi.com/v1beta1/instances"


def request_zero_instance(tag: str = "pytidb-tutorial") -> dict:
    """TiDB Cloud Zero のインスタンスを作成して接続情報を返す"""
    resp = requests.post(ZERO_API, json={"tag": tag}, timeout=30)
    resp.raise_for_status()
    return resp.json()["instance"]


instance = request_zero_instance(tag="pytidb-hybrid")
conn = instance["connection"]
claim_url = instance.get("claimInfo", {}).get("claimUrl", "(取得失敗)")
expires_at = instance.get("expiresAt", "(取得失敗)")

print("=== プロビジョニング完了 ===")
print(f"Host     : {conn['host']}")
print(f"User     : {conn['username']}")
print(f"Expires  : {expires_at}")
print()
print("インスタンスは 30 日後に自動削除されます。")
print("保持したい場合は claim URL を開いてください:")
print(claim_url)


## 2. 商品カタログテーブル

`name` は全文検索用、`description` はベクトル検索用 (自動埋め込み) にします。


In [ ]:
from pytidb import TiDBClient
from pytidb.datatype import TEXT
from pytidb.embeddings import EmbeddingFunction
from pytidb.schema import Field, FullTextField, TableModel

db = TiDBClient.connect(
    host=conn["host"],
    port=4000,
    username=conn["username"],
    password=conn["password"],
    database="shop_demo",
    ensure_db=True,
)

_embed = EmbeddingFunction(model_name="tidbcloud_free/amazon/titan-embed-text-v2")


class Product(TableModel):
    __tablename__ = "products"
    __table_args__ = {"extend_existing": True}

    id: int = Field(primary_key=True)
    category: str = Field()
    price: int = Field()
    name: str = FullTextField(fts_parser="MULTILINGUAL")
    description: str = Field(sa_type=TEXT)
    description_vec: list[float] = _embed.VectorField(source_field="description")


table = db.create_table(schema=Product, if_exists="overwrite")
print("テーブル準備OK:", table.table_name)


## 3. サンプル商品を投入


In [ ]:
PRODUCTS = [
    ("electronics", 12800, "ワイヤレスノイズキャンセリングイヤホン",
     "周囲の騒音を打ち消しながら音楽や通話を楽しめる完全ワイヤレス型のイヤホン。通勤や旅行で集中したい人向け。"),
    ("electronics", 28900, "コンパクトドローン 4K カメラ付き",
     "折りたたんでカバンに入る小型ドローン。屋外で空撮を楽しみたい初心者から中級者に。"),
    ("electronics", 9800,  "スマートポータブル加湿器",
     "USB 給電の小型加湿器。オフィスや枕元で乾燥を防ぎ、静音設計で在宅勤務に最適。"),
    ("kitchen",     4980,  "圧力調理マルチポット",
     "圧力・蒸し・炒めができる一台多役の調理器。忙しい平日の夕食づくりを 30 分で終わらせたい人向け。"),
    ("kitchen",     1980,  "ステンレスコーヒードリッパー",
     "紙フィルター不要の金属メッシュドリッパー。豆の油分まで抽出できて、コーヒーを毎日淹れる人におすすめ。"),
    ("kitchen",     3480,  "電動ミル付きペッパーグラインダー",
     "ボタン一つでスパイスを挽ける電動式グラインダー。料理の仕上げに挽きたての香りを楽しめる。"),
    ("outdoor",     15800, "軽量登山テント 2 人用",
     "総重量 1.9kg の軽量テント。2 人が快適に泊まれるサイズで、縦走やキャンプツーリングに最適。"),
    ("outdoor",     6980,  "防水リュック 30L",
     "完全防水素材を使った 30L のリュック。雨天の通勤や川沿いのアウトドアで中身を守る。"),
    ("outdoor",     2980,  "折りたたみチタンマグカップ",
     "薄くて軽いチタン製のマグカップ。キャンプや登山で温かい飲み物をすぐに楽しめる。"),
    ("fashion",     4280,  "メリノウール ベースレイヤー",
     "冬のアウトドアにも普段着にも使えるメリノウール製の長袖インナー。臭いを抑える天然素材。"),
    ("fashion",     8900,  "撥水ライトシェルジャケット",
     "急な雨や風を防ぐ軽量シェル。自転車通勤やハイキングで羽織るのに便利なアウター。"),
    ("fashion",     1280,  "シンプルコットンTシャツ",
     "肉厚のコットンを使ったユニセックスのTシャツ。普段使いのローテに入れやすいベーシック。"),
]

table.bulk_insert([
    Product(id=i + 1, category=c, price=p, name=n, description=d)
    for i, (c, p, n, d) in enumerate(PRODUCTS)
])
print(f"投入完了: {table.rows()} 件")


## 4. ベクトル単独 vs 全文単独 を先に見る

同じクエリ「雨の日の通勤が快適になるもの」で比較します。
ベクトル検索は意味の近いものを、全文検索は「雨」「通勤」など単語一致を重視します。


In [ ]:
QUERY = "雨の日の通勤が快適になるもの"

print("[ベクトル検索]")
for r in table.search(QUERY, search_type="vector").text_column("description").limit(5).to_pydantic():
    print(f"  sim={r.similarity_score:.3f}  {r.hit.name}")

print("\n[全文検索]")
for r in table.search(QUERY, search_type="fulltext").text_column("name").limit(5).to_pydantic():
    print(f"  ms={r.match_score:.3f}  {r.hit.name}")


## 5. ハイブリッド検索 (既定: RRF)

`search_type="hybrid"` を指定すると、ベクトル結果と全文結果を **Reciprocal Rank Fusion (RRF)** で結合します。
どちらのランクでも上位に登場するドキュメントが優先されます。


In [ ]:
results = (
    table.search(QUERY, search_type="hybrid")
    .text_column("name")
    .vector_column("description_vec")
    .limit(5)
    .to_pydantic()
)
print(f"[hybrid (RRF)] Query: {QUERY}")
for r in results:
    print(f"  score={r.score:.4f}  [{r.hit.category}] {r.hit.name}")


## 6. ハイブリッド検索 (weighted)

`.fusion(method="weighted", vs_weight=..., fts_weight=...)` で重み付き結合に切り替えます。
ベクトル寄りにしたい場合は `vs_weight` を大きく、キーワード寄りにしたい場合は `fts_weight` を大きくします。


In [ ]:
results = (
    table.search(QUERY, search_type="hybrid")
    .text_column("name")
    .vector_column("description_vec")
    .fusion(method="weighted", vs_weight=0.7, fts_weight=0.3)
    .limit(5)
    .to_pydantic()
)
print(f"[hybrid (weighted, vs=0.7/fts=0.3)] Query: {QUERY}")
for r in results:
    print(f"  score={r.score:.4f}  [{r.hit.category}] {r.hit.name}")


## 7. カテゴリフィルタと組み合わせる

ハイブリッド検索も `.filter()` と併用できます。


In [ ]:
results = (
    table.search("山でも普段でも使える", search_type="hybrid")
    .text_column("name")
    .vector_column("description_vec")
    .filter({"category": {"$in": ["outdoor", "fashion"]}})
    .limit(5)
    .to_pydantic()
)
for r in results:
    print(f"  score={r.score:.4f}  [{r.hit.category}] {r.hit.name}")


## 追加実験

- `vs_weight=0.3, fts_weight=0.7` に変えて、結果がキーワード寄りになることを確認
- クエリを英語 (`"cheap coffee gear"`) にして、ベクトルと全文の差を感じる
- RRF の `k` 係数 (`.fusion(method="rrf", k=40)`) で並びがどう変わるかテスト
